# 01_data_preprocessing.ipynb

This notebook preprocesses NYC Yellow Taxi data using Polars and builds taxi demand data in 3-minute intervals.

## 1. Import required libraries

In [2]:
import polars as pl
import os
from datetime import datetime


## 2. Load data (lazy loading)

Load all monthly Parquet files in the `data/raw` directory using lazy loading.
The actual dataset is not included here, but the code assumes files exist under `data/raw`.

In [3]:
# Guidance for the case where `data/raw` does not exist or is empty
# Download real data and place it under `data/raw`.
# Example: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

raw_data_path = "../data/raw/*.parquet"

# Check if files exist (required for real execution)
# import glob
# if not glob.glob(raw_data_path):
#     print("Error: No parquet files found in data/raw. Please download the data first.")
# else:
#     print("Parquet files found. Proceeding with lazy loading.")

lf = pl.scan_parquet(raw_data_path)
print("LazyFrame created. Schema preview (first 5 columns):")
print(lf.collect_schema())


LazyFrame created. Schema preview (first 5 columns):
Schema({'DOLocationID': Int64, 'PULocationID': Int64, 'RatecodeID': Int64, 'VendorID': Int64, 'congestion_surcharge': Float64, 'extra': Float64, 'fare_amount': Float64, 'improvement_surcharge': Float64, 'mta_tax': Float64, 'passenger_count': Int64, 'payment_type': Int64, 'store_and_fwd_flag': String, 'tip_amount': Float64, 'tolls_amount': Float64, 'total_amount': Float64, 'tpep_dropoff_datetime': Datetime(time_unit='us', time_zone=None), 'tpep_pickup_datetime': Datetime(time_unit='us', time_zone=None), 'trip_distance': Float64, 'Airport_fee': Float64, 'trip_id': Int64})


## 3. Data preprocessing

This section performs the following steps:
1. **Column normalization**: If `Airport_fee` exists, rename it to `airport_fee` and fill nulls with `0.0`.
2. **Date and condition filtering**: Select rows from `2023-01-01` to `2025-11-30` by `tpep_pickup_datetime`, and keep rows with `passenger_count > 0` and `trip_distance > 0`.
3. **Demand aggregation**: Truncate `tpep_pickup_datetime` to 3-minute buckets (`3m`), then group by `pickup_time` and `PULocationID` and count rows as `demand`.

In [4]:
processed_lf = lf.with_columns(
    # Normalize `Airport_fee` to `airport_fee` and fill nulls with `0.0`
    pl.col("Airport_fee").fill_null(0.0).alias("airport_fee")
).select(
    # Select required columns and cast pickup datetime for date-range filtering
    pl.col("tpep_pickup_datetime").cast(pl.Datetime).alias("pickup_datetime"),
    "PULocationID",
    "passenger_count",
    "trip_distance",
    "airport_fee" # Use normalized `airport_fee`
).filter(
    (pl.col("pickup_datetime") >= datetime(2023, 1, 1))
    & (pl.col("pickup_datetime") <= datetime(2025, 11, 30))
    & (pl.col("passenger_count") > 0)
    & (pl.col("trip_distance") > 0)
).with_columns(
    # Truncate time in 3-minute intervals
    pl.col("pickup_datetime").dt.truncate("3m").alias("pickup_time")
).group_by(["pickup_time", "PULocationID"]).agg(
    # Aggregate demand by 3-minute interval and location
    pl.len().alias("demand")
).sort(["pickup_time", "PULocationID"])

print("Processed LazyFrame created. The result will be written to parquet in the next step.")


Processed LazyFrame created. The result will be written to parquet in the next step.


## 4. Save output

Save the preprocessed and aggregated data to `data/processed/aggregated_demand_3min.parquet`.

In [5]:
# Execute LazyFrame and materialize as an eager DataFrame
print("Collecting aggregated data...")
aggregated_df = processed_lf.collect()

# Upsample and fill missing time slots with `0` to build a complete time series
print("Upsampling to fill missing time buckets...")
complete_df = (
    aggregated_df
    .upsample(time_column="pickup_time", every="3m", group_by="PULocationID")
    .with_columns([
        pl.col("PULocationID").forward_fill(),
        pl.col("demand").fill_null(0).cast(pl.UInt32) # Fill missing demand with `0`
    ])
)

# Save final artifact
output_path = "../data/processed/aggregated_demand_3min.parquet"
print(f"Saving complete time series data to {output_path}...")
complete_df.write_parquet(output_path)

print("Data processing complete and saved.")
print(f"Data saved to {output_path}")

Upsampling to fill missing time buckets...
Saving complete time series data to ../data/processed/aggregated_demand_3min.parquet...
Data processing complete and saved.
Data saved to ../data/processed/aggregated_demand_3min.parquet


## 5. Inspect saved data (optional)

In [6]:
try:
    check_df = pl.read_parquet(output_path)
    print("Successfully re-read saved parquet file. Head:")
    print(check_df.head())
except Exception as e:
    print(f"Error reading saved file: {e}")


Successfully re-read saved parquet file. Head:
shape: (5, 3)
┌─────────────────────┬──────────────┬────────┐
│ pickup_time         ┆ PULocationID ┆ demand │
│ ---                 ┆ ---          ┆ ---    │
│ datetime[μs]        ┆ i64          ┆ u32    │
╞═════════════════════╪══════════════╪════════╡
│ 2023-01-01 00:00:00 ┆ 170          ┆ 2      │
│ 2023-01-01 00:03:00 ┆ 170          ┆ 3      │
│ 2023-01-01 00:06:00 ┆ 170          ┆ 7      │
│ 2023-01-01 00:09:00 ┆ 170          ┆ 5      │
│ 2023-01-01 00:12:00 ┆ 170          ┆ 5      │
└─────────────────────┴──────────────┴────────┘
